# Приложение Д. Эффективная настройка параметров с помощью LoRA

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # Для предобученных весов OpenAI
        "pandas"      # Загрузка наборов данных
       ]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

matplotlib версия: 3.10.9
numpy версия: 2.4.6
tiktoken версия: 0.13.0
torch версия: 2.12.0
tensorflow версия: 2.21.0
pandas версия: 3.0.3


## Д.1 Введение в LoRA

- Низкоранговая адаптация (Low-rank adaptation (LoRA)) — это метод машинного обучения, который модифицирует предобученную модель для лучшего соответствия конкретному, часто меньшему набору данных путём настройки лишь небольшого низкорангового подмножества параметров модели
- Этот подход важен, поскольку он позволяет эффективно дообучать большие модели на данных, специфичных для конкретной задачи, значительно снижая вычислительные затраты и время, необходимые для тонкой настройки

- Предположим, у нас есть большая весовая матрица $W$ для заданного слоя
- Во время обратного распространения мы изучаем матрицу $\Delta W$, которая содержит информацию о том, насколько мы хотим обновить исходные веса, чтобы минимизировать функцию потерь в процессе обучения
- При обычном обучении и тонкой настройке обновление весов определяется следующим образом:

$$W_{\text{updated}} = W + \Delta W$$

- Метод LoRA, предложенный [Hu et al.](https://arxiv.org/abs/2106.09685), предлагает более эффективную альтернативу вычислению обновлений весов $\Delta W$ путём изучения его аппроксимации, $\Delta W \approx AB$
- Другими словами, в LoRA мы имеем следующее, где $A$ и $B$ — две малые весовые матрицы:

$$W_{\text{updated}} = W + AB$$

- Рисунок ниже иллюстрирует эти формулы для полной тонкой настройки и LoRA рядом друг с другом

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-e_compressed/lora-1.webp" width="800px">

- Изображения полной тонкой настройки и LoRA на рисунке выше выглядят немного иначе, чем формулы, которые были приведены ранее
- Это связано с дистрибутивным законом умножения матриц: нам не нужно складывать веса с обновлёнными весами, а можно держать их раздельно
- Например, если $x$ — это входные данные, то для обычной тонкой настройки мы можем записать следующее:

$$x (W+\Delta W) = x W + x \Delta W$$

- Аналогично, для LoRA мы можем записать следующее:

$$x (W+A B) = x W + x A B$$

- Тот факт, что мы можем держать весовые матрицы LoRA отдельно, делает LoRA особенно привлекательной
- На практике это означает, что нам вообще не нужно изменять веса предобученной модели, так как мы можем применять матрицы LoRA на лету
- После настройки набора данных и загрузки модели мы реализуем LoRA в коде, чтобы сделать эти концепции менее абстрактными